# Parking Occupancy Classification

Academic project developed for PSI3471 – Fundamentos de Sistemas Eletrônicos Inteligentes at Escola Politécnica da Universidade de São Paulo.

The project develops a Deep Learning model to classify parking spaces as available or occupied from images.

In [ ]:
# ==============================
# Exercicio Computacional
# Pedro Madeira Tergolino           NUSP - 13831589
# ==============================

In [ ]:
TEST = "data/test"
TRAIN = "data/test"

In [ ]:
import os
import glob
import cv2
import numpy as np
import matplotlib.pyplot as plt
import xml.etree.ElementTree as ET

import tensorflow as tf

from tensorflow.keras.models import Sequential

from tensorflow.keras.layers import (
    Conv2D,
    MaxPooling2D,
    Flatten,
    Dense,
    Dropout,
    BatchNormalization,
    Activation
)

In [ ]:
train_imgs = sorted(glob.glob(os.path.join(TRAIN, "**", "*.jpg"), recursive=True))
train_xmls = sorted(glob.glob(os.path.join(TRAIN, "**", "*.xml"), recursive=True))

print(f"Imagens: {len(train_imgs)}")
print(f"XMLs:    {len(train_xmls)}")

In [ ]:
# =====================
#  leitura das imagens
# =====================

def read_xml(xml_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()

    parking_spaces = []

    for space in root.findall("space"):

        # Ignora vagas sem informação de ocupação
        if "occupied" not in space.attrib:
            continue

        space_id = int(space.attrib["id"])
        occupied = int(space.attrib["occupied"])

        contour = space.find("contour")

        points = []

        for point in contour.findall("point"):
            x = int(point.attrib["x"])
            y = int(point.attrib["y"])
            points.append((x, y))

        parking_spaces.append({
            "id": space_id,
            "occupied": occupied,
            "points": points
        })

    return parking_spaces


In [ ]:
# =====================
#  desenhando as vagas
# =====================

def draw_parking_spaces(image, spaces):

    img = image.copy()

    for space in spaces:

        pts = np.array(space["points"], dtype=np.int32)

        color = (0,255,0) if space["occupied"] == 0 else (255,0,0)

        cv2.polylines(
            img,
            [pts],
            isClosed=True,
            color=color,
            thickness=2
        )

    return img

In [ ]:
img = cv2.imread(train_imgs[0])

# OpenCV lê em BGR
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

spaces = read_xml(train_xmls[0])

result = draw_parking_spaces(img, spaces)

plt.figure(figsize=(16,9))
plt.imshow(result)
plt.axis("off")
plt.show()

In [ ]:
def distance(p1, p2):
    return np.linalg.norm(np.array(p1) - np.array(p2))

def get_space_size(points):

    p1, p2, p3, p4 = points

    width_top = distance(p1, p2)
    width_bottom = distance(p4, p3)

    height_left = distance(p1, p4)
    height_right = distance(p2, p3)

    width = int(max(width_top, width_bottom))
    height = int(max(height_left, height_right))

    return width, height

In [ ]:
space = spaces[0]

w, h = get_space_size(space["points"])

print(w, h)

In [ ]:
def draw_points(image, points):

    img = image.copy()

    for i, (x, y) in enumerate(points):

        # desenha o ponto
        cv2.circle(img, (x, y), 4, (255, 0, 0), -1)

        # escreve o índice
        cv2.putText(
            img,
            str(i + 1),
            (x + 4, y - 4),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (255, 255, 0),
            2
        )

    return img

In [ ]:
img = cv2.imread(train_imgs[0])
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

space = spaces[71]

img2 = draw_points(img, space["points"])

plt.figure(figsize=(16,9))
plt.imshow(img2)
plt.axis("off")
plt.show()

In [ ]:
def get_space_size(points):

    bottom_left, top_left, top_right, bottom_right = points

    width_top = distance(top_left, top_right)
    width_bottom = distance(bottom_left, bottom_right)

    height_left = distance(top_left, bottom_left)
    height_right = distance(top_right, bottom_right)

    width = int(max(width_top, width_bottom))
    height = int(max(height_left, height_right))

    return width, height

In [ ]:
def extract_space(image, points):

    # Ordem do XML
    bottom_left, top_left, top_right, bottom_right = points

    # Calcula tamanho da vaga
    width, height = get_space_size(points)

    # Pontos de origem
    src = np.float32([
        top_left,
        top_right,
        bottom_right,
        bottom_left
    ])

    # Pontos de destino
    dst = np.float32([
        [0, 0],
        [width - 1, 0],
        [width - 1, height - 1],
        [0, height - 1]
    ])

    # Matriz da homografia
    M = cv2.getPerspectiveTransform(src, dst)

    # Warp
    warped = cv2.warpPerspective(
        image,
        M,
        (width, height)
    )

    return warped

In [ ]:
img = cv2.imread(train_imgs[0])
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

space = spaces[71]

vaga = extract_space(img, space["points"])

plt.figure(figsize=(6,4))
plt.imshow(vaga)
plt.axis("off")
plt.show()

In [ ]:
# =====================
# construindo o dataset
# =====================
def build_dataset(image_paths, xml_paths):

    X = []
    y = []

    for img_path, xml_path in zip(image_paths, xml_paths):

        # Lê imagem
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # Lê XML
        spaces = read_xml(xml_path)

        # Percorre todas as vagas
        for space in spaces:

            vaga = extract_space(img, space["points"])

            vaga = vaga.astype(np.float32) / 255.0

            X.append(vaga)
            y.append(space["occupied"])

    return np.array(X), np.array(y)

In [ ]:
X_train, y_train = build_dataset(train_imgs, train_xmls)

print(X_train.shape)
print(y_train.shape)

In [ ]:
unique, counts = np.unique(y_train, return_counts=True)

for u, c in zip(unique, counts):
    print(f"Classe {u}: {c}")

In [ ]:
import random

plt.figure(figsize=(12,12))

for i in range(16):

    idx = random.randint(0, len(X_train)-1)

    plt.subplot(4,4,i+1)

    plt.imshow(X_train[idx])

    plt.title(f"Classe {y_train[idx]}")

    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
print(X_train.dtype)

print(X_train.min())

print(X_train.max())

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.20,
    random_state=42,
    stratify=y_train
)

In [ ]:
print("Treino:", X_train.shape)
print("Validação:", X_val.shape)

print()

print("Treino labels:", y_train.shape)
print("Validação labels:", y_val.shape)

In [ ]:
def show_distribution(name, labels):

    unique, counts = np.unique(labels, return_counts=True)

    print(name)

    for u,c in zip(unique,counts):
        print(f"Classe {u}: {c}")

    print()

In [ ]:
show_distribution("Treino", y_train)

show_distribution("Validação", y_val)

In [ ]:
model = Sequential()

# Entrada
model.add(
    Conv2D(
        filters=32,
        kernel_size=(3,3),
        padding="same",
        input_shape=(64,64,3)
    )
)

model.add(BatchNormalization())
model.add(Activation("relu"))
model.add(MaxPooling2D(pool_size=(2,2)))

# Segunda camada
model.add(
    Conv2D(
        filters=64,
        kernel_size=(3,3),
        padding="same"
    )
)

model.add(BatchNormalization())
model.add(Activation("relu"))
model.add(MaxPooling2D(pool_size=(2,2)))

# Terceira camada
model.add(
    Conv2D(
        filters=128,
        kernel_size=(3,3),
        padding="same"
    )
)

model.add(BatchNormalization())
model.add(Activation("relu"))
model.add(MaxPooling2D(pool_size=(2,2)))

# Classificador
model.add(Flatten())

model.add(Dense(128))
model.add(Activation("relu"))

model.add(Dropout(0.5))

model.add(Dense(1, activation="sigmoid"))

In [ ]:
model.summary()

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=5e-4
    ),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.AUC(name="auc")
    ]
)

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

classes = np.unique(y_train)

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

class_weights = dict(zip(classes, weights))

print(class_weights)

In [ ]:
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint
)

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

checkpoint = ModelCheckpoint(
    "best_model.keras",
    monitor="val_loss",
    save_best_only=True
)

In [ ]:
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=32,
    class_weight=class_weights,
    callbacks=[early_stop, checkpoint]
)

In [ ]:
plt.figure(figsize=(12,5))

# Accuracy
plt.subplot(1,2,1)

plt.plot(history.history["accuracy"], label="Treino")
plt.plot(history.history["val_accuracy"], label="Validação")

plt.xlabel("Época")
plt.ylabel("Accuracy")
plt.title("Accuracy")
plt.legend()

# Loss
plt.subplot(1,2,2)

plt.plot(history.history["loss"], label="Treino")
plt.plot(history.history["val_loss"], label="Validação")

plt.xlabel("Época")
plt.ylabel("Loss")
plt.title("Loss")
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
y_prob = model.predict(X_val)

In [ ]:
y_pred = (y_prob > 0.5).astype(int)

In [ ]:
from sklearn.metrics import confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay

cm = confusion_matrix(y_val, y_pred)

disp = ConfusionMatrixDisplay(cm)

disp.plot(cmap="Blues")
plt.show()

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(
    y_val,
    y_pred,
    target_names=["Livre","Ocupada"]
))

In [ ]:
errors = np.where(y_pred.flatten() != y_val)[0]

print(len(errors))

In [ ]:
plt.figure(figsize=(12,8))

for i, idx in enumerate(errors):

    plt.subplot(2,4,i+1)

    plt.imshow(X_val[idx])

    plt.title(
        f"Real:{y_val[idx]}\nPred:{y_pred[idx][0]}"
    )

    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
test_imgs = []
test_xmls = []

for day in sorted(os.listdir(TEST)):

    folder = os.path.join(TEST, day)

    imgs = sorted(glob.glob(os.path.join(folder, "*.jpg")))
    xmls = sorted(glob.glob(os.path.join(folder, "*.xml")))

    test_imgs.extend(imgs)
    test_xmls.extend(xmls)

print("Imagens:", len(test_imgs))
print("XMLs:", len(test_xmls))

In [ ]:
X_test, y_test = build_dataset(test_imgs, test_xmls)

print(X_test.shape)
print(y_test.shape)

In [ ]:
y_prob = model.predict(X_test)

y_pred = (y_prob > 0.5).astype(int)

In [ ]:
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay

print(classification_report(
    y_test,
    y_pred,
    target_names=["Livre","Ocupada"]
))

cm = confusion_matrix(y_test, y_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Livre","Ocupada"]
)

disp.plot(cmap="Blues")
plt.show()

## Notes

- The dataset is not included in this repository.
- Update the `data/train` and `data/test` paths if running locally.
- This notebook is presented as an academic project and preserves the original modeling approach.